# Romanian-Transformers Model Evaluation

Use this colab to evaluate _tranformer model performance_. We currently support:

*   [**Named Entity Recognition**](https://github.com/dumitrescustefan/ronec), based on RONECv2
*   [**Part of Speech Tagging**](https://github.com/dumitrescustefan/ro-pos-tagger), based on UD's Ro-RRT dataset
*   [**Semantic Textual Simiarity**](https://github.com/dumitrescustefan/RO-STS), based on RO-STS
*   [**Emotion Detection in Tweets**](https://github.com/Alegzandra/RED-Romanian-Emotions-Dataset), based on REDv2
*   [**Perplexity**](https://github.com/dumitrescustefan/wiki-ro), based on wiki-ro, only for generative models

**How to use:**
1. Choose the model in the dropdown below.
2. Choose the task
3. Choose the number of iterations (how many times to run the same task and average over)
4. Run all cells ($\color{red}{\text{Ctrl+F9}}$); you will see the averaged results printed at the bottom, as well as saved as jsons in each task's respective folder.

** Note: Use a GPU runtime and a browser addon (like Colab Auto Reconnect) to keep this session open - some tasks, with 5 iterations, might take some hours to complete.
This is the official script used to measure model performance on
the [Romanian-Transformers repo](https://github.com/dumitrescustefan/Romanian-Transformers).


---



In [ ]:
#@title Evaluation parameters

model = 'dumitrescustefan/bert-base-romanian-cased-v1' #@param ["dumitrescustefan/gpt-neo-romanian-780m", "dumitrescustefan/bert-base-romanian-cased-v1","dumitrescustefan/bert-base-romanian-uncased-v1", "racai/distilbert-base-romanian-cased", "readerbench/RoGPT2-base", "readerbench/RoGPT2-medium", "readerbench/RoGPT2-large", "xlm-roberta-base", "bert-base-multilingual-cased", "bert-base-multilingual-uncased", "readerbench/RoBERT-small", "readerbench/RoBERT-base", "readerbench/RoBERT-large"] {allow-input: true}
task = 'Named Entity Recognition' #@param ["Named Entity Recognition", "POS Tagging", "Emotion Detection in Tweets", "Semantic Textual Similarity", "CLM Perplexity"]
iterations = '1' #@param ["1", "2", "3", "5"]

print("\nWe are going to eval \033[92m{}\033[0m on the \033[92m{}\033[0m task for \033[92m{}\033[0m iteration(s).\n".format(model, task, iterations))

import torch
if not torch.cuda.is_available():
  print(f"\033[101m*** Please use a GPU-enabled colab! ***\033[0m")
else:
  print(f"\nRunning on a \033[92m{torch.cuda.get_device_name(0)}\033[0m with \033[92m{torch.cuda.get_device_properties(0).total_memory/1024/1024/1024:.0f}GB\033[0m RAM.")


We are going to eval dumitrescustefan/bert-base-romanian-cased-v1 on the Named Entity Recognition task for 5 iteration(s).


Running on a Tesla T4 with 15GB RAM.


##### Code section, run every cell automatically with Ctrl+F9

In [ ]:
def eval_ner(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/ronec.git
  !pip3 install -r ronec/evaluate/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd ronec/evaluate && python evaluate.py --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches --experiment_iterations $iterations --model_name $model

  return None

def eval_pos(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/ro-pos-tagger.git
  !pip3 install -r ro-pos-tagger/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd ro-pos-tagger/model && python evaluate_pos_tagger.py --experiment_iterations $iterations --model_name $model --batch_size=$batch_size

  return None

def eval_redv2(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/ronec.git
  !pip3 install -r ronec/evaluate/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd ronec/evaluate && python eval_ronec_v2.py --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches --experiment_iterations $iterations --model_name $model

  return None

def eval_sts(model, iterations, batch_size, accumulate_grad_batches):
  print("\033[92mPreparing environment ...\033[0m")
  !git clone https://github.com/dumitrescustefan/RO-STS.git
  !pip3 install -r RO-STS/baseline-models/requirements.txt -q

  print("\033[92mRunning task ...\033[0m")
  !cd RO-STS/baseline-models && python transformer_model.py --experiment_iterations $iterations --model_name $model --batch_size=$batch_size --accumulate_grad_batches=$accumulate_grad_batches

  return None

def eval_ppl(model_id, stride, batch_size):
  batch_size = 1
  print("\033[92mPreparing environment ...\033[0m")
  !pip3 install -q transformers datasets
  import torch, os
  import torch.nn.functional as F
  from tqdm import tqdm
  from transformers import AutoModelForCausalLM, AutoTokenizer
  from datasets import load_dataset
  if not os.path.exists("wiki-ro"):
    !git clone https://github.com/dumitrescustefan/wiki-ro.git
    !cd wiki-ro/corpus && cat wiki-ro-split-* > wiki-ro.zip && unzip wiki-ro.zip
  ppl_dataset = load_dataset('text', data_files={'test': 'wiki-ro/corpus/wiki.txt.test'})

  print("\033[92mRunning task ...\033[0m")
  model = AutoModelForCausalLM.from_pretrained(model_id).to("cuda")
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  try:
    max_len = model.config.n_positions
  except:
    max_len = 512  # default in case model config does not have n_position set

  device = model.device
  encodings = tokenizer("\n\n".join(ppl_dataset["test"]["text"]), return_tensors="pt")
  text_len = encodings.input_ids.size(1)
  lls = []

  for i in tqdm(range(0, text_len, batch_size * stride)):
      begin_locs, end_locs, trg_lens = [], [], []
      for j in range(batch_size):
          j = i + j * stride
          if j >= text_len:
              break
          begin_loc = max(j + stride - max_len, 0)
          end_loc = min(j + stride, text_len)
          trg_len = end_loc - j  # may be different from stride on last loop

          begin_locs.append(begin_loc)
          end_locs.append(end_loc)
          trg_lens.append(trg_len)

      input_ids = [encodings.input_ids[:, b:e] for b, e in zip(begin_locs, end_locs)]
      target_end_locs = [sen.size(-1) for sen in input_ids]
      input_ids = [
          F.pad(sen, (0, max_len - sen.size(-1)), "constant", 0) for sen in input_ids
      ] # we dont need attention mask as long as these padded token is not involved in loss calculation
      input_ids = torch.stack(input_ids, dim=1).squeeze(0).to(device)

      target_ids = torch.ones_like(input_ids) * -100 # -100 is the default ingore_index value in torch.nn.CrossEntropyLoss
      for i, (b, e) in enumerate(zip(trg_lens, target_end_locs)):
          labels = input_ids[i, -b:e].clone()
          target_ids[i, -b:e] = labels

      with torch.no_grad():
          outputs = model(input_ids, labels=target_ids)
          log_likelihood = outputs["loss"] * sum(trg_lens)

      if not torch.isnan(log_likelihood):
        lls.append(log_likelihood)

  ppl = torch.exp(sum(torch.stack(lls) / end_locs[-1]))
  print(f"\nModel {model_id} perplexity on Ro-Wiki test split: {ppl}")

# configs
batch_size, accumulate_grad_batches = 8, 1
if "-large" in model or "-medium" in model:
  batch_size = 1
  accumulate_grad_batches = 8

In [ ]:
# run
if task == "Named Entity Recognition":
  eval_ner(model, iterations, batch_size, accumulate_grad_batches)
if task == "POS Tagging":
  eval_pos(model, iterations, batch_size, accumulate_grad_batches)
if task == "Emotion Detection in Tweets":
  eval_redv2(model, iterations, batch_size, accumulate_grad_batches)
if task == "Semantic Textual Similarity":
  eval_sts(model, iterations, batch_size, accumulate_grad_batches)
if task == "CLM Perplexity":
  eval_ppl(model, stride=512, batch_size=1)

Preparing environment ...
Cloning into 'ronec'...
remote: Enumerating objects: 502, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 502 (delta 59), reused 61 (delta 24), pack-reused 387
Receiving objects: 100% (502/502), 24.89 MiB | 15.61 MiB/s, done.
Resolving deltas: 100% (218/218), done.
     |████████████████████████████████| 5.5 MB 26.1 MB/s 
     |████████████████████████████████| 798 kB 58.8 MB/s 
     |████████████████████████████████| 7.6 MB 52.7 MB/s 
     |████████████████████████████████| 182 kB 67.1 MB/s 
     |████████████████████████████████| 529 kB 73.0 MB/s 
     |████████████████████████████████| 87 kB 7.3 MB/s 
Running task ...
Using a random seed.
Loading data...
	Dataset contains 31 BIO2 classes: ['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-NAT_REL_POL', 'I-NAT_REL_POL', 'B-EVENT', 'I-EVENT', 'B-LANGUAGE', 'I-LANGUAGE', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B